# Validating a custom writer

Any `ecgprocess.writer.BaseWriter` subclass can be supplied as the `writer` argument of `ECGTable.write_ecg`. 

This notebook builds a small custom writer, checks it with `validate_writer` before running it against real data, and shows what a failed check looks like. 
See [Custom writers](../../custom_writers.rst) for the full contract. 

This notebook is validation-focused and does not repeat the built-in writer tour; see the [Pluggable output writers](writer.ipynb) notebook for that.

In [1]:
import json
import os
import tempfile
import numpy as np
from ecgprocess.errors import InputValidationError
from ecgprocess.writer import BaseWriter, WriteContext, WriterValidator

## Define a custom writer

`JsonWriter` bundles each chunk into a single `.json` file using only the standard library, it honours the export flags, writes into `ctx.target`, and derives its filename
through `self.file_names`.

In [2]:
class JsonWriter(BaseWriter):
    """Writes each chunk of extracted ECG data to one .json."""
    FILE_EXT = '.json'
    def write(self, meta_list, wave_list, median_list, ctx):
        # the fourth standard name is the bundled-output slot
        _, _, _, ecg_file, _ = self.file_names(chunk_num=ctx.chunk_num)
        record = {'identifiers': list(ctx.key_list)}
        if ctx.e_meta:
            record['metadata'] = meta_list
        if ctx.e_wave:
            record['waveforms'] = self._signals(wave_list)
        if ctx.e_median:
            record['medianbeats'] = self._signals(median_list)
        with open(os.path.join(ctx.target, ecg_file), 'w') as fh:
            json.dump(record, fh)
    @staticmethod
    def _signals(signal_list):
        """Map numpy arrays to lists, dropping absent leads."""
        return [
            {lead: signal.tolist()
             for lead, signal in signals.items()
             if signal is not None}
            for signals in signal_list
        ]

## validate_writer: a compliant writer passes silently

`validate_writer` is a light-touch, data-free pre-flight check: it
inspects the class or instance once and never writes files. It accepts
either the class itself or a configured instance.

In [3]:
WriterValidator(JsonWriter)
WriterValidator(JsonWriter())
print('JsonWriter is compliant.')

JsonWriter is compliant.


## A broken writer: the collected multi-violation report

`validate_writer` collects every violation and raises a single
`InputValidationError` listing all of them, so one run reports
everything that needs fixing.

In [4]:
class BrokenWriter(BaseWriter):
    # a non-empty FILE_EXT must start with a dot
    FILE_EXT = 'json'
    def file_names(self, chunk_num=None):
        # the contract requires the standard 5-tuple of filenames
        return ('metadata.json', 'waveforms.json', 'medianbeats.json')
    def write(self):
        # write must accept (meta_list, wave_list, median_list, ctx)
        return None

try:
    WriterValidator(BrokenWriter)
except InputValidationError as err:
    print(err)

the supplied writer failed validation with 3 violation(s):
- `write` cannot accept the four data arguments (meta_list, wave_list, median_list, ctx): too many positional arguments.
- a non-empty `FILE_EXT` must start with a dot; got `json`.
- `file_names()` must return 5 filenames, got 3.


## Understanding the contract with WriterValidator.describe()

`WriterValidator.describe()` provides a human-readable checklist of the `BaseWriter` contract requirements, listing all required methods, optional overrides, and `WriteContext` fields.

## Running the compliant writer against synthetic data

A minimal `WriteContext` and two small synthetic records exercise
`JsonWriter.write` directly, without going through `ECGTable`.

In [ ]:
print(WriterValidator.describe())

In [5]:
meta_list = [
    {'unique identifier': 'ecg_1', 'sampling frequency (original)': 500.0},
    {'unique identifier': 'ecg_2', 'sampling frequency (original)': 500.0},
]
wave_list = [
    {'I': np.arange(5.0), 'II': np.arange(5.0) + 1.0},
    {'I': np.arange(5.0) + 2.0, 'II': None},
]

with tempfile.TemporaryDirectory() as temp_dir:
    ctx = WriteContext(
        e_meta=True, e_wave=True, e_median=False,
        key_list=['ecg_1', 'ecg_2'], target=temp_dir, chunk_num=None,
    )
    JsonWriter().write(meta_list, wave_list, [], ctx)
    print(sorted(os.listdir(temp_dir)))
    with open(os.path.join(temp_dir, 'ecg_data.json')) as fh:
        written = json.load(fh)
    print(json.dumps(written, indent=2))

['ecg_data.json']
{
  "identifiers": [
    "ecg_1",
    "ecg_2"
  ],
  "metadata": [
    {
      "unique identifier": "ecg_1",
      "sampling frequency (original)": 500.0
    },
    {
      "unique identifier": "ecg_2",
      "sampling frequency (original)": 500.0
    }
  ],
  "waveforms": [
    {
      "I": [
        0.0,
        1.0,
        2.0,
        3.0,
        4.0
      ],
      "II": [
        1.0,
        2.0,
        3.0,
        4.0,
        5.0
      ]
    },
    {
      "I": [
        2.0,
        3.0,
        4.0,
        5.0,
        6.0
      ]
    }
  ]
}
